In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
X = np.memmap("X_AllStepsR_f32_norm.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
Y = np.load("Y_AllStepsR.npy")
XRef = np.load("XRef_AllStepsR_f32_norm.npy")

### 1. Try to train a classifier for X/Y

In [ ]:
BATCH_SIZE = 32
VAL_SPLIT = 0.05

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True, flat=True):
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch, 0:1]

            if flat:
                X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
            yield X_, Y_

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
A, B = next(gen_train)

In [ ]:
import inception
model = inception.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), # CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="acc") # tf.keras.metrics.CategoricalAccuracy(name="acc")
    ]
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(id_train)//BATCH_SIZE//2, # Smaller "fake epochs" for faster feedback
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
# model.save_weights('best_model.weights.h5')
model.load_weights('best_model.weights.h5')

In [ ]:
model.evaluate(gen_valid, steps=len(id_valid)//BATCH_SIZE)